# Агрономические модели (RandomForest) — анализ готовых артефактов

**Модели уже обучены** и загружаются из `agro-ml-service/models/`.

| Модель | Артефакты | Задача |
|--------|-----------|--------|
| **Crop Recommender** | `crop_model.pkl`, `crop_label_encoder.pkl` | Рекомендация культуры по почве и климату |
| **Fertilizer Recommender** | `fertilizer_model.pkl`, `fertilizer_encoders.pkl` | Рекомендация удобрения |
| **Pesticide Recommender** | `pesticide_model.pkl`, `pesticide_encoders.pkl` | Рекомендация пестицида |
| **Disease Predictor** | `disease_model.pkl`, `disease_risk_model.pkl`, `disease_encoders.pkl` | Прогноз болезни и уровня риска |

---

## Структура ноутбука
1. Загрузка артефактов и обзор моделей
2. Crop Recommender — feature importance + примеры инференса
3. Fertilizer Recommender — примеры инференса
4. Pesticide Recommender — примеры инференса
5. Disease Predictor — feature importance + примеры инференса

## 1. Загрузка артефактов

In [ ]:
import warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from pathlib import Path

warnings.filterwarnings('ignore')

# Ноутбук лежит в agro-ml-service/notebooks/
NOTEBOOK_DIR = Path('.').resolve()
SERVICE_DIR  = NOTEBOOK_DIR.parent   # agro-ml-service/
MODELS_DIR   = SERVICE_DIR / 'models'

print(f'Модели: {MODELS_DIR.resolve()}')
print()
for fname in sorted(MODELS_DIR.glob('*.pkl')):
    size_mb = fname.stat().st_size / 1024 / 1024
    print(f'  {fname.name:45}  {size_mb:>6.1f} MB')

# ── Загрузка всех моделей ──────────────────────────────────────────────────────
model_crop     = joblib.load(MODELS_DIR / 'crop_model.pkl')
le_crop        = joblib.load(MODELS_DIR / 'crop_label_encoder.pkl')

model_fert     = joblib.load(MODELS_DIR / 'fertilizer_model.pkl')
fert_encoders  = joblib.load(MODELS_DIR / 'fertilizer_encoders.pkl')

model_pest     = joblib.load(MODELS_DIR / 'pesticide_model.pkl')
pest_encoders  = joblib.load(MODELS_DIR / 'pesticide_encoders.pkl')

model_disease  = joblib.load(MODELS_DIR / 'disease_model.pkl')
model_risk     = joblib.load(MODELS_DIR / 'disease_risk_model.pkl')
disease_enc    = joblib.load(MODELS_DIR / 'disease_encoders.pkl')

print('\nВсе модели загружены успешно')

## 2. Crop Recommender

**Признаки:** N, P, K, temperature, humidity, ph, rainfall  
**Таргет:** культура (label)  
**Используется в сервисе:** `POST /crop/recommend` → `soilCompatibilityScore` в рекомендациях севооборота

In [ ]:
# Классы и количество деревьев
print(f'Деревьев      : {model_crop.n_estimators}')
print(f'Классов       : {len(le_crop.classes_)}')
print(f'Классы        : {list(le_crop.classes_)}')
print(f'Признаков     : {model_crop.n_features_in_}')

In [ ]:
# Feature importance crop model
feature_cols = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
crop_imp = pd.Series(model_crop.feature_importances_, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 4))
crop_imp.plot(kind='barh', ax=ax, color='steelblue', alpha=0.85)
ax.set_title('Важность признаков (crop model, Gini)')
ax.set_xlabel('Feature importance')
plt.tight_layout()
plt.show()

print('Топ признаков:')
for feat, imp in crop_imp.sort_values(ascending=False).items():
    print(f'  {feat:15} {imp:.4f}')

In [ ]:
# Примеры инференса — рекомендация культуры
examples = [
    {'label': 'Омская обл. (чернозём, умеренно)',
     'N': 50, 'P': 35, 'K': 120, 'temperature': 19.5, 'humidity': 62.0, 'ph': 6.9, 'rainfall': 210.0},
    {'label': 'Алтай (засуха)',
     'N': 30, 'P': 20, 'K': 80,  'temperature': 23.0, 'humidity': 45.0, 'ph': 7.2, 'rainfall': 130.0},
    {'label': 'Западная Сибирь (кислая почва)',
     'N': 70, 'P': 50, 'K': 150, 'temperature': 17.0, 'humidity': 72.0, 'ph': 5.3, 'rainfall': 280.0},
]

for ex in examples:
    X = pd.DataFrame([{k: v for k, v in ex.items() if k != 'label'}])
    probs = model_crop.predict_proba(X)[0]
    top_idx = np.argsort(probs)[::-1]
    print(f'\n>>> {ex["label"]}')
    for i in top_idx[:4]:
        bar = '█' * int(probs[i] * 40)
        print(f'  {le_crop.classes_[i]:20} {probs[i]:.1%}  {bar}')

In [ ]:
## 3. Fertilizer Recommender

**Признаки:** crop_type, soil_type, deficiency_level (все категориальные)  
**Таргет:** название удобрения  
**Используется в сервисе:** `POST /agro/fertilizer/recommend`

In [ ]:
# Классы модели удобрений
le_fert = fert_encoders['fertilizer']
print(f'Деревьев  : {model_fert.n_estimators}')
print(f'Удобрений : {len(le_fert.classes_)}')
print(f'Классы    : {list(le_fert.classes_)}')
print()
print('Входные классы:')
for k in ['crop_type', 'soil_type', 'deficiency_level']:
    print(f'  {k}: {list(fert_encoders[k].classes_)}')

In [ ]:
def safe_encode(le, val):
    val = str(val)
    if val not in set(le.classes_):
        val = le.classes_[0]
    return int(le.transform([val])[0])

def recommend_fertilizer(crop_type, soil_type, deficiency_level):
    X = np.array([[
        safe_encode(fert_encoders['crop_type'],        crop_type),
        safe_encode(fert_encoders['soil_type'],        soil_type),
        safe_encode(fert_encoders['deficiency_level'], deficiency_level),
    ]])
    probs = model_fert.predict_proba(X)[0]
    top_idx = np.argsort(probs)[::-1]
    print(f'  {crop_type} | {soil_type} | дефицит: {deficiency_level}')
    for i in top_idx[:3]:
        print(f'    {fert_encoders["fertilizer"].classes_[i]:30} {probs[i]:.1%}')

print('=== Рекомендации удобрений ===')
recommend_fertilizer('wheat',     'chernozem', 'Nitrogen')
print()
recommend_fertilizer('sunflower', 'loamy',     'Phosphorus')
print()
recommend_fertilizer('corn',      'solonetz',  'Potassium')

In [ ]:
## 4. Pesticide Recommender

**Признаки:** crop, pest_type, intensity  
**Таргет:** название пестицида  
**Используется в сервисе:** `POST /agro/pesticide/recommend`

# Классы модели пестицидов
le_pest = pest_encoders['pesticide']
print(f'Деревьев  : {model_pest.n_estimators}')
print(f'Пестицидов: {len(le_pest.classes_)}')
print(f'Классы    : {list(le_pest.classes_)}')
print()
print('Входные классы:')
for k in ['crop', 'pest_type', 'intensity']:
    print(f'  {k}: {list(pest_encoders[k].classes_)}')

In [ ]:
def recommend_pesticide(crop, pest_type, intensity):
    X = np.array([[
        safe_encode(pest_encoders['crop'],      crop),
        safe_encode(pest_encoders['pest_type'], pest_type),
        safe_encode(pest_encoders['intensity'], intensity),
    ]])
    probs = model_pest.predict_proba(X)[0]
    top_idx = np.argsort(probs)[::-1]
    print(f'  {crop} | {pest_type} | интенсивность: {intensity}')
    for i in top_idx[:3]:
        print(f'    {pest_encoders["pesticide"].classes_[i]:30} {probs[i]:.1%}')

print('=== Рекомендации пестицидов ===')
recommend_pesticide('wheat',     'fungal',  'high')
print()
recommend_pesticide('corn',      'insect',  'medium')
print()
recommend_pesticide('sunflower', 'weed',    'low')

In [ ]:
## 5. Disease Predictor

**Признаки:** crop, growth_stage (cat) + temperature, humidity, rainfall (num)  
**Таргеты:** disease (болезнь) + risk_level (low/medium/high)  
**Используется в сервисе:** `POST /disease/predict`

In [ ]:
# Классы disease модели
print(f'DiseaseClassifier:')
print(f'  Деревьев  : {model_disease.n_estimators}')
print(f'  Болезней  : {len(disease_enc["disease"].classes_)}')
print(f'  Классы    : {list(disease_enc["disease"].classes_)}')
print()
print(f'RiskClassifier:')
print(f'  Деревьев  : {model_risk.n_estimators}')
print(f'  Уровни    : {list(disease_enc["risk_level"].classes_)}')
print()
print('Входные классы:')
for k in ['crop', 'growth_stage']:
    print(f'  {k}: {list(disease_enc[k].classes_)}')

In [ ]:
# Feature importance — disease model
feature_names_dis = ['crop', 'growth_stage', 'temperature', 'humidity', 'rainfall']
dis_imp = pd.Series(model_disease.feature_importances_, index=feature_names_dis).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 3))
dis_imp.plot(kind='barh', ax=ax, color='steelblue', alpha=0.85)
ax.set_title('Важность признаков (disease model, Gini)')
ax.set_xlabel('Feature importance')
plt.tight_layout()
plt.show()

In [ ]:
def predict_disease(crop, growth_stage, temperature, humidity, rainfall):
    crop_enc   = safe_encode(disease_enc['crop'],         crop)
    stage_enc  = safe_encode(disease_enc['growth_stage'], growth_stage)
    X = np.array([[crop_enc, stage_enc, temperature, humidity, rainfall]])

    d_probs = model_disease.predict_proba(X)[0]
    r_probs = model_risk.predict_proba(X)[0]
    top_d   = np.argsort(d_probs)[::-1]
    risk    = disease_enc['risk_level'].classes_[np.argmax(r_probs)]

    print(f'  {crop} | {growth_stage} | T={temperature} H={humidity}% rain={rainfall}mm')
    print(f'  Уровень риска: {risk} ({r_probs.max():.1%})')
    for i in top_d[:3]:
        print(f'    {disease_enc["disease"].classes_[i]:35} {d_probs[i]:.1%}')

print('=== Прогноз болезней ===')
predict_disease('spring_wheat', 'flowering',   22.0, 84.0, 95.0)
print()
predict_disease('corn',         'vegetative',  26.0, 55.0, 40.0)
print()
predict_disease('sunflower',    'ripening',    18.0, 78.0, 60.0)

print('Артефакты в', MODELS_DIR.resolve())
print()
for fname in sorted(MODELS_DIR.glob('*.pkl')):
    size_mb = fname.stat().st_size / 1024 / 1024
    print(f'  {fname.name:45}  {size_mb:>6.1f} MB')
print()
print('Инференс: agro-ml-service/app/predictors/agro_predictor.py')
print('Переобучение CLI: python agro-ml-service/train/train_agro_models.py')

In [ ]:
# TODO: Delete this cell

In [ ]:
# TODO: Delete this cell

In [ ]:
# TODO: Delete this cell

In [ ]:
# TODO: Delete this cell

In [ ]:
# TODO: Delete this cell

# TODO: Delete this cell

In [ ]:
# TODO: Delete this cell

In [ ]:
# TODO: Delete this cell

In [ ]:
# TODO: Delete this cell

In [ ]:
# TODO: Delete this cell

In [ ]:
# TODO: Delete this cell

In [ ]:
# TODO: Delete this cell

In [ ]:
# TODO: Delete this cell

In [ ]:
# TODO: Delete this cell

# TODO: Delete this cell

In [ ]:
# TODO: Delete this cell

In [ ]:
# TODO: Delete this cell

In [ ]:
# TODO: Delete this cell